### Ingest Fact Data into Bronze Layer

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, BooleanType
import pyspark.sql.functions as F
from datetime import datetime
import os

In [0]:
catalog_name = 'ecommerce'

#### Setting up current date variable to Archive in Date folder

In [0]:
archive_date = datetime.now().strftime("%Y-%m-%d")
print("archive_date : ",archive_date)

### Order Items

In [0]:
order_items_schema = StructType([
    StructField("dt",                 StringType(), True),
    StructField("order_ts",           StringType(), True),
    StructField("customer_id",        StringType(), True),
    StructField("order_id",           StringType(), True),
    StructField("item_seq",           StringType(), True),
    StructField("product_id",         StringType(), True),
    StructField("quantity",           StringType(), True),
    StructField("unit_price_currency",StringType(), True),
    StructField("unit_price",         StringType(), True),
    StructField("discount_pct",       StringType(), True),
    StructField("tax_amount",         StringType(), True),
    StructField("channel",            StringType(), True),
    StructField("coupon_code",        StringType(), True),
])

In [0]:
# Load data using the schema defined
raw_data_path = "/Volumes/ecommerce/source_data/raw/order_items/landing/*.csv"

df = spark.read.option("header", "true").option("delimiter", ",").schema(order_items_schema).csv(raw_data_path) \
    .withColumn("_source_file", F.col("_metadata.file_path")) \
    .withColumn("ingest_timestamp", F.current_timestamp())

In [0]:
display(df.limit(5))

In [0]:
archive_base_path = f"/Volumes/ecommerce/source_data/archive/{archive_date}/order_items/"
print("archive_base_path : ",archive_base_path)

try:
    # Append raw orders data to bronze Delta table
    # Enable Delta's Change Data Feed for incremental processing
    (
        df.write
          .format("delta")
          .option("delta.enableChangeDataFeed", "true")
          .mode("append")
          .option("mergeSchema", "true")
          .saveAsTable(f"{catalog_name}.bronze.brz_order_items")
    )

    # Create archive folder
    dbutils.fs.mkdirs(archive_base_path)

    # Get all processed source files
    source_files = (
        df.select("_source_file")
          .distinct()
          .collect()
    )

    # Archive each file
    for row in source_files:
        source_file = row["_source_file"]
        file_name = os.path.basename(source_file)
        destination_file = f"{archive_base_path}{file_name}"
        dbutils.fs.mv(source_file, destination_file)

    print("Delta load and file archival completed successfully.")

except Exception as e:
    print(f"Pipeline failed: {str(e)}")
    raise